# Evaluating Observed Flood Extent from the GFM against Forecast Flood Extent from the EFAS or GloFAS Rapid Flood Mapping layer

With this notebook, we show how to use the new GFM STAC service to
derive the maximum flood extent for a flood event, extract the corresponding
forecast product from the EFAS or GloFAS Rapid Flood Mapping (RFM) layer and perform
a simple evaluation.

This example notebook is structured as follows:

0. Install and import the necessary Python packages
1. Extract GFM data using the GFM STAC service (ensemble flood extent, reference water mask, exclusion mask)
2. Save the data as a Zarr store for later use (recommended, but optional)
3. Mask the data and save it as GeoTIFF (optional)
4. Download the forecast flood extent from the EFAS or GloFAS Rapid Flood Mapping layer
5. Compare the observed and forecast flood extent
6. Visualize the outputs in a GIS application

We present an example analysis of the flooding that occurred in Portugal and Spain in February 2026.

Please note you must have an EFAS account to access EFAS data.
GloFAS data is publicly accessible and does not require an account.

## 0. Install and import necessary Python modules

The following cells were successfully tested with Python 3.11.10. Run the following cell to install all necessary Python modules.
The `--user` flag installs the packages into your home directory, which is useful on shared environments such as JupyterHub.

In [ ]:
!pip install pystac_client==0.9.0 odc-stac==0.5.2 rioxarray==0.19.0 geopandas==1.1.3 'zarr>=3' --user

In [2]:
# Some necessary imports

import shutil
from datetime import datetime, timedelta
from pathlib import Path
from zipfile import ZipFile

import geopandas as gpd
import numpy as np
import pyproj
import rasterio
import requests
import rioxarray  # noqa
import xarray as xr
from odc import stac as odc_stac
from pystac_client import Client
from rasterio.features import rasterize
from shapely.geometry import box

## 1. Extract GFM Data using the GFM STAC service

Firstly define the GFM product you wish to download.

Then define the date of the flood forecast you want to evaluate and the coordinates of the area of
interest (AOI). The time range of the GFM observations is derived from the forecast date. The coordinates of the AOI can be retrieved either via the GFM Portal (portal.gfm.eodc.eu) or a tool like bboxfinder.com.

These parameters are passed to the GFM STAC service to find the available data.

### Use Case: Portugal February 2026

Run the following cell if you want to check out the Portugal February 2026 use case.

In [3]:
use_case_name = "Portugal_Feb_2026"

# Define bounding box (West, South, East, North) - Portugal, Lisbon area
aoi = box(-9.179077, 38.613651, -8.375702, 39.448919)

# Date of the flood forecast (RFM) to compare against
forecast_date = datetime(2026, 2, 1)

# The EFAS medium-range forecast covers the next 10 days, so we compare against
# all GFM observations within the same period
forecast_horizon = timedelta(days=10)
time_range = (forecast_date, forecast_date + forecast_horizon)

The Rapid Flood Mapping layer shows the maximum flood extent expected over the full forecast period
(10 days for the EFAS medium-range forecast). To compare like with like, we use the GFM observations
from the same period, i.e. from `forecast_date` to `forecast_date + forecast_horizon`.
You can change `forecast_date` to evaluate a different forecast (e.g. `datetime(2026, 2, 7)`).

### Extract GFM Data

In [4]:
# Define the output path, where you want to save temporary data and final results
output_path = Path("./gfm_rfm_evaluation") / use_case_name
output_path.mkdir(parents=True, exist_ok=True)

# EODC STAC API URL
eodc_catalog = Client.open("https://stac.eodc.eu/api/v1")

# Define search query using pystac_client
search = eodc_catalog.search(
    max_items=1000,
    collections=["GFM"],
    bbox=aoi.bounds,
    datetime=time_range
)

# Get all items
items = search.item_collection()
if not items:
    raise RuntimeError("No items found — check your AOI, time range, or collection name.")

print(f"Found {len(items)} items matching filter criteria.")

# Retrieve the coordinate reference system (CRS) from one of the found items
crs_equi7 = pyproj.CRS.from_wkt(items[0].properties["proj:wkt2"])

Found 65 items matching filter criteria.


### Load the returned data
Now the returned data are loaded into an xarray dataset in the Equi7Grid projection.

We define the asset names we want to use in this notebook. In our case, that is `ensemble_flood_extent`, `reference_water_mask` and `exclusion_mask`.

Some returned items do not cover the AOI, because the geometry of a GFM STAC item is the whole Equi7Grid tile and not the actual footprint of the Sentinel-1 scene. After loading, we therefore remove all time steps which only contain nodata within the AOI.

Note that the data is loaded directly into memory. If your kernel crashes due to out of memory issues, reduce the size of the AOI or the time range.

In [5]:
# Define the asset names you want to use
asset_names = ["ensemble_flood_extent", "reference_water_mask", "exclusion_mask"]

# Retrieve the resolution of the data
resolution = items[0].properties.get("gsd", 20)  # fallback to 20m if gsd missing

# Load the data into memory. Items from the same day are grouped into one time step
raw_data = odc_stac.load(
    items,
    bbox=aoi.bounds,  # Define the bounding box for the area of interest
    crs=crs_equi7,  # Set the coordinate reference system
    bands=asset_names,  # Specify the bands to load
    resolution=resolution,  # Set the resolution of the data
    dtype="uint8",  # Define the data type
    groupby="solar_day",
)

# The STAC item geometry covers the whole Equi7Grid tile, so some time steps
# only contain nodata (=255) within the AOI. Remove these empty time steps
has_data = (raw_data["ensemble_flood_extent"] != 255).any(dim=("x", "y"))
raw_data = raw_data.sel(time=has_data)
print(f"{raw_data.sizes['time']} time steps with valid data within the AOI.")

4 time steps with valid data within the AOI.


## 2. Save the data as a Zarr store for later use (recommended, but optional)
Once the above cell has finished, we advise to save the data on disk for later use.
With that, you do not need to download the same data every time you start this notebook.

### Save the loaded data as a Zarr store

In [6]:
# Save the "raw" data as Zarr data store for later use
zarr_store = output_path / "gfm_data.zarr"
shutil.rmtree(zarr_store, ignore_errors=True)  # Remove existing Zarr store if it exists
raw_data.to_zarr(zarr_store, consolidated=False);

### OPTIONAL: Load the data from the Zarr store once saved

If you have previously saved the GFM data as a Zarr store, uncomment and run the following cell to load it directly. Otherwise, you can skip it.

In [ ]:
# OPTIONAL: If you want to read the data from the Zarr store again, uncomment the following lines.
# This is useful if you want to restart the notebook without reloading the data from the STAC API.

# zarr_store = output_path / "gfm_data.zarr"

# # Read data from Zarr data store once saved
# raw_data = xr.open_zarr(zarr_store, chunks=None, consolidated=False).load()

# # Zarr is currently not writing CRSs into data stores. Add it again after
# # opening the data store
# raw_data.rio.write_crs(crs_equi7, inplace=True)

## 3. Mask data

We mask the data to exclude invalid values such as nodata (=255) and non-flood (=0).

Additionally, we binarize all layers over the time range: a pixel becomes 1 if it was flagged
at least once, otherwise it stays 0. For the flood extent, this gives the maximum flood extent.

In [7]:
# Only include data which is not nodata and not 0
nodata = 255

# Binarize all layers: 1 if the pixel was flagged at least once in the time range, else 0
data = xr.Dataset(
    {
        var: ((raw_data[var] != nodata) & (raw_data[var] != 0)).any(dim="time").astype("uint8")
        for var in asset_names
    }
)
data.rio.write_crs(crs_equi7, inplace=True)

<xarray.Dataset> Size: 89MB
Dimensions:                (y: 5851, x: 5056)
Coordinates:
  * y                      (y) float64 47kB 1.241e+06 1.241e+06 ... 1.124e+06
  * x                      (x) float64 40kB 3.004e+06 3.004e+06 ... 3.105e+06
    spatial_ref            int64 8B 0
Data variables:
    ensemble_flood_extent  (y, x) uint8 30MB 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0
    reference_water_mask   (y, x) uint8 30MB 1 1 1 1 1 1 1 1 ... 0 0 0 0 0 0 0 0
    exclusion_mask         (y, x) uint8 30MB 0 0 0 0 0 0 0 0 ... 1 1 1 1 1 1 1 1

### OPTIONAL: Save GFM data layers as GeoTIFF

After masking, we can save a GeoTIFF file for each GFM layer for easy visualization in e.g. QGIS.

In [8]:
# OPTIONAL: If you want to save the processed GFM data as GeoTIFF files, you can use the following code.

rio_options = {
    "compress": "LZW",
    "tiled": True,
    "blockxsize": 512,
    "blockysize": 512,
}

# Save as raster in Equi7Grid
for var in asset_names:
    data[var].rio.to_raster(output_path / f"{var}_equi7.tif", **rio_options)

## 4. Download Forecasted Flood Extent from the EFAS or GloFAS Rapid Flood Mapping layer

The EFAS Rapid Flood Mapping (RFM) layer is produced as follows
(see [EFAS flood impact forecasts](https://european-flood.emergency.copernicus.eu/flood-impact-forecasts)):
* Whenever a flood event greater than the 10-year return period is forecasted, the return period of the
  maximum discharge over the full forecast period is computed for each river grid cell
* For these river sections, the flood-prone areas are delineated using a catalogue of flood inundation maps
* The resulting flood extent has a spatial resolution of 90 m (GFM: 20 m)
* It represents the *unprotected scenario*, i.e. it does not consider the role of flood defences such as dikes

Specify the data `source` (`"efas"` or `"glofas"`). The forecast date was already defined in the use case cell above.

For EFAS, you will also need a web token, which you can generate by logging in to
https://european-flood.emergency.copernicus.eu/en/efas-web-services and copying
the web token shown in the table. Note: the token needs to be regenerated after every use.

GloFAS data is publicly accessible and does not require a token.

In [9]:
source = "efas"  # Set to "efas" or "glofas"
fcast_str = forecast_date.strftime("%Y%m%d00")  # date of the forecast as string in YYYYMMDDHH format

# EFAS requires a token; GloFAS does not
# Only needed when source == "efas", the token must be regenerated after every use
efas_token = "<YOUR_EFAS_TOKEN>"
efas_token = "d21d79386f9328e83729ae62dcd6bd8c"

date_str = forecast_date.strftime("%Y-%m-%dT00:00Z")
base_url = f"https://european-flood.emergency.copernicus.eu/api/fms/download/{source}/RapidFloodMapping/{date_str}"
url_rfm = f"{base_url}?token={efas_token}" if source == "efas" else base_url

#### Download the forecast flood extent file

This will save a zip file into the output folder defined above
(`gfm_rfm_evaluation/<use_case_name>`). An ESRI shapefile containing the forecast flood
extent will be extracted from this zip file into the same folder.

In [10]:
response = requests.get(url_rfm)
response.raise_for_status()

out_zip = output_path / f"RFM_{fcast_str}.zip"
with open(out_zip, "wb") as file:
    file.write(response.content)

with ZipFile(out_zip, "r") as zip_ref:
    zip_ref.extractall(output_path)

#### Load the Forecasted Flood Extent file

The forecast flood extent shapefile is loaded as a GeoPandas
GeoDataFrame. It is then clipped to the extent of the AOI
which you defined previously.

In [11]:
rfm_file = output_path / f"FloodMask{fcast_str}.shp"
rfm_data = gpd.read_file(rfm_file)

# Clip the Rapid Flood Mapping to your AOI
rfm_intersect = gpd.clip(rfm_data, gpd.GeoDataFrame(index=[0], crs="EPSG:4326", geometry=[aoi]))

if rfm_intersect.empty:
    raise RuntimeError("No RFM data found within the AOI — check your bounding box or forecast date.")

#### Rasterize the forecast flood extent onto the GFM grid

To compare both datasets pixel by pixel, the RFM polygons are reprojected to the Equi7Grid
projection and burned into a raster with the same extent and resolution as the GFM data
(1 = forecast flooded, 0 = not flooded).

In [12]:
gfm_flood = data["ensemble_flood_extent"].values

# Use the geotransform of the GFM data, so both datasets share the same grid
transform = data.rio.transform()

# Reproject the RFM polygons to the Equi7Grid projection of the GFM data
rfm_intersect_equi7 = rfm_intersect.to_crs(crs_equi7)

# Assign a value of 1 to every RFM polygon
shapes = ((geom, 1) for geom in rfm_intersect_equi7.geometry)

# Rasterize the geometries into a numpy array
rfm_raster = rasterize(
    shapes,
    out_shape=gfm_flood.shape,
    transform=transform,
    fill=0,
    dtype=np.uint8,
)

## 5. Compare the Observed and Forecast Flood Extent

Use a contingency table approach to classify each grid cell as one of the following:
1. Hit - where flooding is present in both the GFM & RFM forecast
2. False alarm - flooding only present in the RFM forecast
3. Miss - flooding only present in the GFM observations
4. Correct negative - no flooding present in either GFM or RFM data

We exclude areas which are either in the reference water mask or the exclusion mask.

From this classification, we compute the following performance scores:
1. CSI (Critical Success Index), range 0-1 where 1 = perfect agreement between forecast and observations
2. False Alarm Ratio (FAR), range 0-1 where 0 is optimal, shows the fraction of grid cells forecasted as flooded which were not observed as flooded
3. Hit Rate (HR), range 0-1 where 1 is optimal, shows the fraction of observed flooded grid cells which were correctly forecasted

In [13]:
# Compare GFM and RFM data

gfm_em = data["exclusion_mask"].values
gfm_rwm = data["reference_water_mask"].values

# Only evaluate pixels outside the reference water mask and the exclusion mask
valid = (gfm_rwm == 0) & (gfm_em == 0)

hits = (gfm_flood == 1) & (rfm_raster == 1) & valid
false_alarms = (gfm_flood == 0) & (rfm_raster == 1) & valid
misses = (gfm_flood == 1) & (rfm_raster == 0) & valid

# Compute the total number of hits, false alarms and misses
tp = np.count_nonzero(hits)
fp = np.count_nonzero(false_alarms)
fn = np.count_nonzero(misses)

csi = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else float("nan")  # critical success index (CSI)
far = fp / (tp + fp) if (tp + fp) > 0 else float("nan")  # false alarm ratio (FAR)
hr = tp / (tp + fn) if (tp + fn) > 0 else float("nan")  # hit rate (HR)

print(f"CSI: {csi:.2f}")
print(f"False Alarm Ratio: {far:.2f}")
print(f"Hit Rate: {hr:.2f}")

# Create a grid which shows whether each grid cell is a hit, miss or false alarm
eval_grid = np.zeros(gfm_flood.shape, dtype=np.uint8)
eval_grid[hits] = 1
eval_grid[misses] = 2
eval_grid[false_alarms] = 3

CSI: 0.53
False Alarm Ratio: 0.43
Hit Rate: 0.88


### OPTIONAL: Save the evaluation results as GeoTIFF file

The resulting raster will have values of 1 for hits, 2 for misses and 3 for false alarms.

In [14]:
# OPTIONAL: If you want to save the evaluation results as GeoTIFF file, you can use the following code.

profile = {
    "driver": "GTiff",
    "height": rfm_raster.shape[0],
    "width": rfm_raster.shape[1],
    "count": 1,
    "dtype": "uint8",
    "crs": crs_equi7,
    "transform": transform,
    "compress": "LZW",
}

# Save RFM data as raster
with rasterio.open(output_path / "rfm_data_equi7.tif", "w", **profile) as dst:
    dst.write(rfm_raster, 1)

# Save RFM GFM evaluation results as raster
with rasterio.open(output_path / "rfm_gfm_eval_equi7.tif", "w", **profile) as dst:
    dst.write(eval_grid, 1)

## 6. Visualize outputs in your favourite GIS application (e.g. QGIS, ArcGIS)

![RFM GFM evaluation results in QGIS](./gfm_rfm_evaluation/Portugal_Feb_2026/rfm_gfm_eval_equi7.png)

## Conclusions

This notebook shows how you can download specific GFM data
using the GFM STAC service and the EFAS or GloFAS Rapid Flood Mapping forecast
layer and compare the data, with the idea of evaluating the
performance of the forecast layer.

For the Portugal February 2026 example, the EFAS forecast of 1 February 2026 compared against the GFM
observations of the following 10 days reached a CSI of 0.53, a False Alarm Ratio of 0.43 and a Hit Rate of 0.88.
In general it could be seen that:
* The forecast captured most of the observed flooding (high Hit Rate), but generally
  over-predicted the inundation extent compared to GFM (high False Alarm Ratio)
* This is partly because the RFM layer represents the unprotected scenario, i.e. flood defences
  such as levees and dikes are not considered
* In addition, the RFM layer has a coarser resolution (90 m vs. 20 m for GFM) and is based on a catalogue
  of precomputed flood inundation maps, which gives a smoothed representation of the floodplain topography
* Therefore, you need to be careful regarding the precision for which the forecast information can be used:
    * It can highlight general floodplain areas at risk, but be careful when identifying specific towns/cities affected